In [185]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score,mean_absolute_error
import joblib


In [167]:
df = pd.read_csv("athlete_recovery_synthetic.csv")
print(df.head(5))
print("Dataset shape : ",df.shape)


   Athlete_ID  Day  Day_of_Week  Week  Age Gender  Sport_Type Training_Type  \
0        1000    1            1     1   28   Male  Team Sport          HIIT   
1        1000    2            2     1   28   Male  Team Sport        Cardio   
2        1000    3            3     1   28   Male  Team Sport          HIIT   
3        1000    4            4     1   28   Male  Team Sport        Cardio   
4        1000    5            5     1   28   Male  Team Sport          HIIT   

   Training_Duration_Min  Training_Intensity  Sleep_Duration_Hours  \
0                     46                 7.9                   7.7   
1                     71                 7.0                   7.3   
2                     45                 7.3                   7.7   
3                     86                 7.5                   7.9   
4                     28                 9.2                   6.7   

   Caffeine_Intake_mg Stress_Level  Resting_Heart_Rate  HRV_ms  Mood_Score  \
0                 270     

In [168]:
df.describe()

,Athlete_ID,Day,Day_of_Week,Week,Age,Training_Duration_Min,Training_Intensity,Sleep_Duration_Hours,Caffeine_Intake_mg,Resting_Heart_Rate,HRV_ms,Mood_Score,Muscle_Soreness,Energy_Level,Recovery_Score
count,8379.000000,8379.000000,8379.000000,8379.000000,8379.000000,8379.000000,8378.000000,7775.000000,8379.000000,8379.000000,8379.000000,8379.000000,8379.000000,8379.000000,8379.000000
mean,1149.458647,14.497076,3.998329,2.499821,25.860962,52.789712,6.211506,7.498842,185.348132,57.202888,74.126507,5.177109,5.161953,3.273087,49.061356
std,86.631832,8.078044,2.000417,1.117941,4.531092,21.357704,2.368960,0.693790,112.850208,5.790991,13.981632,1.210072,2.127583,1.679100,27.926347
min,1000.000000,1.000000,1.000000,1.000000,18.000000,10.000000,1.000000,5.000000,0.000000,38.000000,22.000000,1.000000,1.000000,1.000000,0.000000
25%,1074.000000,8.000000,2.000000,2.000000,23.000000,36.000000,4.900000,7.000000,126.000000,53.000000,64.000000,4.400000,3.600000,2.000000,27.800000
50%,1149.000000,14.000000,4.000000,2.000000,26.000000,56.000000,6.600000,7.500000,212.000000,58.000000,74.000000,5.200000,5.200000,3.100000,48.900000
75%,1224.000000,21.000000,6.000000,3.000000,28.000000,69.000000,7.900000,8.000000,268.000000,61.000000,84.000000,6.000000,6.700000,4.300000,69.900000
max,1299.000000,28.000000,7.000000,4.000000,41.000000,115.000000,10.000000,9.500000,400.000000,81.000000,115.000000,9.500000,10.000000,10.000000,100.000000


In [169]:
print((df.isnull().sum()))

Athlete_ID                 0
Day                        0
Day_of_Week                0
Week                       0
Age                        0
Gender                     0
Sport_Type                 0
Training_Type              0
Training_Duration_Min      0
Training_Intensity         1
Sleep_Duration_Hours     604
Caffeine_Intake_mg         0
Stress_Level               0
Resting_Heart_Rate         0
HRV_ms                     0
Mood_Score                 0
Muscle_Soreness            0
Energy_Level               0
Recovery_Score             0
dtype: int64


In [170]:
df["Training_Intensity"] = df["Training_Intensity"].fillna(df["Training_Intensity"].mean())
df["Sleep_Duration_Hours"]= df["Sleep_Duration_Hours"].fillna(df["Sleep_Duration_Hours"].mean())
print("Dataset after replacing null values with mean of respective column : ")
print(df.isnull().sum())
print(df.columns)

Dataset after replacing null values with mean of respective column : 
Athlete_ID               0
Day                      0
Day_of_Week              0
Week                     0
Age                      0
Gender                   0
Sport_Type               0
Training_Type            0
Training_Duration_Min    0
Training_Intensity       0
Sleep_Duration_Hours     0
Caffeine_Intake_mg       0
Stress_Level             0
Resting_Heart_Rate       0
HRV_ms                   0
Mood_Score               0
Muscle_Soreness          0
Energy_Level             0
Recovery_Score           0
dtype: int64
Index(['Athlete_ID', 'Day', 'Day_of_Week', 'Week', 'Age', 'Gender',
       'Sport_Type', 'Training_Type', 'Training_Duration_Min',
       'Training_Intensity', 'Sleep_Duration_Hours', 'Caffeine_Intake_mg',
       'Stress_Level', 'Resting_Heart_Rate', 'HRV_ms', 'Mood_Score',
       'Muscle_Soreness', 'Energy_Level', 'Recovery_Score'],
      dtype='str')


In [171]:
useless_cols = ["Athlete_ID", "Day","Day_of_Week","Week"]
df = df.drop(useless_cols,axis =1)
df

,Age,Gender,Sport_Type,Training_Type,Training_Duration_Min,Training_Intensity,Sleep_Duration_Hours,Caffeine_Intake_mg,Stress_Level,Resting_Heart_Rate,HRV_ms,Mood_Score,Muscle_Soreness,Energy_Level,Recovery_Score
0,28,Male,Team Sport,HIIT,46,7.9,7.7,270,High,59,75,4.3,4.8,5.9,51.1
1,28,Male,Team Sport,Cardio,71,7.0,7.3,258,Low,55,79,5.9,4.0,5.8,63.7
2,28,Male,Team Sport,HIIT,45,7.3,7.7,214,Medium,57,77,5.1,4.0,5.6,71.0
3,28,Male,Team Sport,Cardio,86,7.5,7.9,228,High,61,71,3.0,6.2,2.7,37.2
4,28,Male,Team Sport,HIIT,28,9.2,6.7,0,High,64,62,3.3,7.4,1.1,12.3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8374,28,Female,Strength,Strength,73,6.1,7.8,228,Medium,59,62,5.7,6.5,3.1,31.9
8375,28,Female,Strength,Strength,56,5.2,6.8,0,Low,54,68,6.7,5.5,4.4,36.1
8376,28,Female,Strength,Strength,50,4.9,7.3,167,Low,58,68,6.0,5.1,3.6,29.4
8377,28,Female,Strength,Yoga,36,4.2,7.0,180,High,56,65,4.7,4.3,2.7,31.5


In [172]:
df.columns

Index(['Age', 'Gender', 'Sport_Type', 'Training_Type', 'Training_Duration_Min',
       'Training_Intensity', 'Sleep_Duration_Hours', 'Caffeine_Intake_mg',
       'Stress_Level', 'Resting_Heart_Rate', 'HRV_ms', 'Mood_Score',
       'Muscle_Soreness', 'Energy_Level', 'Recovery_Score'],
      dtype='str')

In [173]:
df['Injury_Risk_Score'] = (
    (100 - df['Recovery_Score']) * 0.4 +  
    (df['Muscle_Soreness'] * 10) * 0.3 +    
    ((8 - df['Sleep_Duration_Hours']).clip(0) * 10) * 0.3 
)

df['Injury_Risk_Score'] = df['Injury_Risk_Score'].clip(0, 100) 

In [174]:
cat_cols = ["Gender", "Sport_Type", "Training_Type","Stress_Level"]

num_cols = df.drop(cat_cols + ['Injury_Risk_Score'], axis=1).columns
print("Categorical columns = ")
print(df[cat_cols])
print("Numerical Columns = ")
print(num_cols)

Categorical columns = 
      Gender  Sport_Type Training_Type Stress_Level
0       Male  Team Sport          HIIT         High
1       Male  Team Sport        Cardio          Low
2       Male  Team Sport          HIIT       Medium
3       Male  Team Sport        Cardio         High
4       Male  Team Sport          HIIT         High
...      ...         ...           ...          ...
8374  Female    Strength      Strength       Medium
8375  Female    Strength      Strength          Low
8376  Female    Strength      Strength          Low
8377  Female    Strength          Yoga         High
8378  Female    Strength          Yoga          Low

[8379 rows x 4 columns]
Numerical Columns = 
Index(['Age', 'Training_Duration_Min', 'Training_Intensity',
       'Sleep_Duration_Hours', 'Caffeine_Intake_mg', 'Resting_Heart_Rate',
       'HRV_ms', 'Mood_Score', 'Muscle_Soreness', 'Energy_Level',
       'Recovery_Score'],
      dtype='str')


In [175]:
df.columns

Index(['Age', 'Gender', 'Sport_Type', 'Training_Type', 'Training_Duration_Min',
       'Training_Intensity', 'Sleep_Duration_Hours', 'Caffeine_Intake_mg',
       'Stress_Level', 'Resting_Heart_Rate', 'HRV_ms', 'Mood_Score',
       'Muscle_Soreness', 'Energy_Level', 'Recovery_Score',
       'Injury_Risk_Score'],
      dtype='str')

In [176]:
df["Injury_Risk_Score"].isnull().sum()

np.int64(0)

In [177]:
X = df.drop(columns = "Injury_Risk_Score")
y = df["Injury_Risk_Score"]
print("Target Column : ")
print(y)
print("Input Column : ")
print(X)

Target Column : 
0       34.86
1       28.62
2       24.50
3       44.02
4       61.18
        ...  
8374    47.34
8375    45.66
8376    45.64
8377    43.30
8378    15.08
Name: Injury_Risk_Score, Length: 8379, dtype: float64
Input Column : 
      Age  Gender  Sport_Type Training_Type  Training_Duration_Min  \
0      28    Male  Team Sport          HIIT                     46   
1      28    Male  Team Sport        Cardio                     71   
2      28    Male  Team Sport          HIIT                     45   
3      28    Male  Team Sport        Cardio                     86   
4      28    Male  Team Sport          HIIT                     28   
...   ...     ...         ...           ...                    ...   
8374   28  Female    Strength      Strength                     73   
8375   28  Female    Strength      Strength                     56   
8376   28  Female    Strength      Strength                     50   
8377   28  Female    Strength          Yoga                

In [178]:
X = pd.get_dummies(X, columns = cat_cols, drop_first=True)

In [179]:
scaler = StandardScaler()
X[num_cols]= scaler.fit_transform(X[num_cols])


In [180]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
print("X_train shape : ",X_train.shape)
print("X_test shape : ",X_test.shape)
print("y_train shape : ",y_train.shape)
print("y_test shape : ",y_test.shape)


X_train shape :  (6703, 23)
X_test shape :  (1676, 23)
y_train shape :  (6703,)
y_test shape :  (1676,)


In [181]:
model = RandomForestRegressor(n_estimators=10,random_state=42)
model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",10
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of

In [182]:
y_pred = model.predict(X_test)
print("First 10 Predicted values = ")
print(y_pred[0:10])
print("First 10 Actual values = ")
print(y_test[0:10])


First 10 Predicted values = 
[57.626      31.86        9.838      53.24669453 19.734      69.02434727
 14.576      54.512      59.962      23.97      ]
First 10 Actual values = 
2865    58.10
8169    31.70
7366     9.60
4857    53.06
5666    19.54
2031    69.10
1056    13.92
2858    54.76
6282    59.78
4038    24.12
Name: Injury_Risk_Score, dtype: float64


In [184]:
print("The r2_score of model is = ",r2_score(y_test, y_pred))
print("The MAE: of model is = ", mean_absolute_error(y_test, y_pred))

The r2_score of model is =  0.9980751861709317
The MAE: of model is =  0.5194323339140046


In [186]:
joblib.dump(model, "injury_risk_model.joblib")
joblib.dump(scaler, "injury_scaler.joblib")
joblib.dump(X.columns.tolist(), "injury_columns.joblib")

print("files saved ")

files saved 
